# Optimize temporal synthetic-data parameters

Goal: find parameters such that IID and cluster-aware TOST conclude equivalence at Δ=1, but temporal-aware TOST (HAC) rejects equivalence at Δ=1.

In [1]:
import os, sys, json
from pathlib import Path

# If this notebook is in the directory that contains `main_module/`, add it to sys.path
# ROOT = Path('.').resolve()
# if str(ROOT) not in sys.path:
#     sys.path.insert(0, str(ROOT))

# print('cwd:', ROOT)

In [2]:
from pyTOST.data_gen.optimize_temporal_params import search, save_best, SearchSpace, evaluate_params

In [4]:
space = SearchSpace(
    n_time_min=200,
    n_time_max=900,
    series_min=1,
    series_max=4,
    rho_min=0.93,
    rho_max=0.999,
    process_sd_min=0.05,
    process_sd_max=1.2,
    obs_sd_min=0.01,
    obs_sd_max=0.3,
    delta_true_min=0.90,
    delta_true_max=0.999,
)

# candidates = []
# for s in range(20):
best, hist = search(
    seed=123,
    n_iter=5000,
    alpha=0.05,
    margin=1.0,
    hac_lags=40,
    space=space,
    verbose_every=1000
)
#     if best.ok_pattern:
#         candidates.append(best)

# sorted(candidates, key=lambda r: r.score)[:3]



[1000/5000] searching best_score=0.001072 eq(iid,clu,tmp)=(False,True,False) CI_tmp=(0.9952221030212823, 1.000935437276471)
[2000/5000] searching best_score=0.001072 eq(iid,clu,tmp)=(False,True,False) CI_tmp=(0.9952221030212823, 1.000935437276471)
[3000/5000] searching best_score=8.36e-05 eq(iid,clu,tmp)=(False,True,False) CI_tmp=(0.9870604087473487, 1.0000350281536945)
Pattern found at iter 3249: score=0


In [ ]:
# for candidate in candidates:
#     print(f'IID: {candidate.eq_iid}\nCluster: {candidate.eq_cluster}\nTemporal: {candidate.eq_temporal}')

In [ ]:
# best = candidates[0]

In [5]:
print(f'IID: {best.eq_iid}\nCluster: {best.eq_cluster}\nTemporal: {best.eq_temporal}')

IID: True
Cluster: True
Temporal: False


In [6]:
print(f'IID: {best.ci_iid}\nCluster: {best.ci_cluster}\ntemporal: {best.ci_temporal}')

IID: (0.9845464412918968, 0.9999318513197847)
Cluster: (0.9898612483658162, 0.9946170442458653)
temporal: (0.9835864549509511, 1.0008918376607305)


In [7]:
from pprint import pprint
pprint(best.params)

{'_hac_lags': 40,
 '_meta': {'effect': {'delta': 0.9954220251498908, 'delta_fn': None},
           'n_time': 364,
           'obs_sd': 0.10556540102750801,
           'process_sd': 0.7253985554828491,
           'rho': 0.9277388275567028,
           'series_per_arm': 3},
 'delta': 0.9954220251498908,
 'n_time': 364,
 'obs_sd': 0.10556540102750801,
 'process_sd': 0.7253985554828491,
 'rho': 0.9277388275567028,
 'seed': 4371,
 'series_per_arm': 3}


In [8]:
out_path = 'best_temporal_params.json'

save_best(best, out_path)

print('Wrote', out_path)

Wrote best_temporal_params.json


In [9]:
import pandas as pd
from pyTOST.engines.iid_tost import IIDTOST
from pyTOST.engines.cluster_tost import ClusterTOST
from pyTOST.engines.temporal_tost import TemporalTOST
from pyTOST.data_gen import synthetic_tost_data

# Rebuild a dataset using the best params and report results at Δ in {1,3,5}
p = dict(best.params)
seed = p.pop('seed', 123)
hac_lags = int(p.pop('_hac_lags', 8))

df_long, meta = synthetic_tost_data.generate_temporal_ar1(**{k:v for k,v in p.items() if k!='_meta'})
wide = df_long.pivot(index='sample_id', columns='arm', values='y').reset_index()
meta2 = df_long[df_long['arm']=='A'][['sample_id','t']].copy()
if 'series_id' in df_long.columns:
    meta2['group_id'] = df_long[df_long['arm']=='A']['series_id'].values
else:
    meta2['group_id'] = 0
df = meta2.merge(wide, on='sample_id')
df['diff'] = df['B'] - df['A']

margins=[1,3,5]
alpha=0.05
res_iid = IIDTOST('diff').fit(df, alpha=alpha, margins=margins)
res_clu = ClusterTOST('diff','group_id').fit(df, alpha=alpha, margins=margins)
res_tmp = TemporalTOST('diff','t', hac_lags=hac_lags).fit(df, alpha=alpha, margins=margins)

display(pd.concat([res_iid.assign(engine='IID'), res_clu.assign(engine='Cluster'), res_tmp.assign(engine='Temporal')], ignore_index=True))
print('meta:', meta)
print('mean(diff):', df['diff'].mean())

,delta,mu_hat,ci_low,ci_high,equivalent,method,df,engine
0,1.0,0.992662,0.985186,1.000138,False,IID OLS (t-CI),1091.0,IID
1,3.0,0.992662,0.985186,1.000138,True,IID OLS (t-CI),1091.0,IID
2,5.0,0.992662,0.985186,1.000138,True,IID OLS (t-CI),1091.0,IID
3,1.0,0.992662,0.985668,0.999655,True,OLS + cluster-robust SE (df=2),2.0,Cluster
4,3.0,0.992662,0.985668,0.999655,True,OLS + cluster-robust SE (df=2),2.0,Cluster
5,5.0,0.992662,0.985668,0.999655,True,OLS + cluster-robust SE (df=2),2.0,Cluster
6,1.0,0.992662,0.985482,0.999841,True,IID mean with Newey–West HAC (lags=40),inf,Temporal
7,3.0,0.992662,0.985482,0.999841,True,IID mean with Newey–West HAC (lags=40),inf,Temporal
8,5.0,0.992662,0.985482,0.999841,True,IID mean with Newey–West HAC (lags=40),inf,Temporal


meta: {'type': 'temporal_ar1', 'n_time': 364, 'series_per_arm': 3, 'rho': 0.9277388275567028, 'process_sd': 0.7253985554828491, 'obs_sd': 0.10556540102750801, 'effect': {'delta': 0.9954220251498908, 'delta_fn': None}}
mean(diff): 0.9926616838505382
